# Phase 2 — Exploratory image preprocessing

This notebook creates diagnostic examples only. Its masks are candidate foreground representations, **not** seed counts or germination labels. Raw images are read only.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import pandas as pd

current_directory = Path.cwd().resolve()
candidates = [current_directory, *current_directory.parents, current_directory / 'seed-germination']
PROJECT_ROOT = next((path for path in candidates if (path / 'src').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Could not find the project root containing src/.')
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import ProjectPaths
from src.image_utils import color_spaces, load_image, split_channels
from src.io_utils import build_inventory, save_csv
from src.preprocessing import MorphologyConfig, SegmentationConfig, apply_morphology, create_foreground_mask, threshold_image

paths = ProjectPaths(root=PROJECT_ROOT)
figure_dir = paths.figures / 'preprocessing'
figure_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
inventory = build_inventory(paths.raw_images)
assert not inventory.empty, 'Add experimental images to data/raw/images/ first.'
assert inventory['is_readable'].all(), 'Resolve unreadable images before preprocessing.'
assert inventory[['width', 'height', 'channels']].drop_duplicates().shape[0] == 1, 'Inspect inconsistent image dimensions/channels.'
representative = inventory.iloc[[0, len(inventory) // 2, len(inventory) - 1]].copy()
representative[['filename', 'timestamp', 'width', 'height', 'channels']]

## ROI assessment

The full frame is deliberately the current candidate analysis region. No tighter ROI is imposed until a seed-containing region is manually validated across early, middle, and late images.

In [ ]:
fig, axes = plt.subplots(1, len(representative), figsize=(16, 5), constrained_layout=True)
for axis, record in zip(axes, representative.itertuples(index=False)):
    image = load_image(Path(record.path), expected_size=(record.width, record.height))
    axis.imshow(color_spaces(image)['rgb'])
    axis.add_patch(patches.Rectangle((0, 0), record.width - 1, record.height - 1, fill=False, edgecolor='lime', linewidth=2))
    axis.set_title(f'{record.timestamp:%Y-%m-%d %H:%M}\nCandidate ROI: full frame')
    axis.axis('off')
fig.savefig(figure_dir / 'roi_assessment.png', dpi=160, bbox_inches='tight')
plt.show()

In [ ]:
# These are exploratory parameters, not validated scientific settings.
candidate_config = SegmentationConfig(
    method='otsu', dark_foreground=True,
    morphology=MorphologyConfig(kernel_size=3, closing_iterations=1),
)
threshold_configs = {
    'fixed_100': SegmentationConfig(method='fixed', fixed_threshold=100),
    'otsu': SegmentationConfig(method='otsu'),
    'adaptive': SegmentationConfig(method='adaptive', adaptive_block_size=51, adaptive_c=5),
}
candidate_config

In [ ]:
def coverage(mask):
    return float((mask == 255).mean() * 100)

diagnostics = []
for record in representative.itertuples(index=False):
    image = load_image(Path(record.path), expected_size=(record.width, record.height))
    spaces = color_spaces(image)
    masks = {name: threshold_image(spaces['gray'], cfg) for name, cfg in threshold_configs.items()}
    threshold_mask, final_mask = create_foreground_mask(image, candidate_config)
    cleaned_mask = apply_morphology(threshold_mask, candidate_config.morphology)
    diagnostics.extend({'timestamp': record.timestamp, 'filename': record.filename, 'method': name, 'foreground_percent': coverage(mask)} for name, mask in masks.items())

    fig, axes = plt.subplots(3, 4, figsize=(15, 10), constrained_layout=True)
    panels = [
        ('Original RGB', spaces['rgb'], None), ('Grayscale', spaces['gray'], 'gray'),
        ('HSV: H', split_channels(spaces['hsv'], 'hsv')['H'], 'hsv'), ('HSV: S', split_channels(spaces['hsv'], 'hsv')['S'], 'gray'),
        ('HSV: V', split_channels(spaces['hsv'], 'hsv')['V'], 'gray'), ('LAB: L', split_channels(spaces['lab'], 'lab')['L'], 'gray'),
        ('LAB: a', split_channels(spaces['lab'], 'lab')['a'], 'coolwarm'), ('LAB: b', split_channels(spaces['lab'], 'lab')['b'], 'coolwarm'),
        ('Otsu candidate threshold', threshold_mask, 'gray'), ('Morphologically cleaned', cleaned_mask, 'gray'),
        ('Final proposed foreground', final_mask, 'gray'), ('Unused panel', None, None),
    ]
    for axis, (title, panel, cmap) in zip(axes.flat, panels):
        axis.set_title(title)
        if panel is not None: axis.imshow(panel, cmap=cmap)
        axis.axis('off')
    fig.suptitle(f'Preprocessing diagnostic — {record.timestamp:%Y-%m-%d %H:%M:%S}', fontsize=14)
    figure_name = f'preprocessing_{record.timestamp:%Y%m%d_%H%M%S}.png'
    fig.savefig(figure_dir / figure_name, dpi=160, bbox_inches='tight')
    plt.show()

diagnostics = pd.DataFrame(diagnostics)
save_csv(diagnostics, paths.tables / 'preprocessing_diagnostics.csv')
diagnostics.pivot(index='timestamp', columns='method', values='foreground_percent')

In [ ]:
fig, axes = plt.subplots(len(representative), 4, figsize=(16, 4 * len(representative)), constrained_layout=True)
for row, record in enumerate(representative.itertuples(index=False)):
    image = load_image(Path(record.path), expected_size=(record.width, record.height))
    gray = color_spaces(image)['gray']
    panels = [('Grayscale', gray), *[(name, threshold_image(gray, config)) for name, config in threshold_configs.items()]]
    for axis, (title, panel) in zip(axes[row], panels):
        axis.imshow(panel, cmap='gray')
        axis.set_title(f'{record.timestamp:%m-%d %H:%M} — {title}')
        axis.axis('off')
fig.savefig(figure_dir / 'threshold_method_comparison.png', dpi=160, bbox_inches='tight')
plt.show()

## Interpretation checklist

Inspect every saved mask against its RGB image before selecting parameters. Prefer temporal consistency and preservation of seed boundaries and thin candidate structures over visual smoothness. Opening/erosion may delete thin structures; defaults therefore apply only one conservative closing step. Test `closing_iterations=0`, other kernel sizes, and background removal before adopting any setting.